In [4]:
# ============================================================
# RISK PULSE - MODEL TRAINING
# CELL 1: IMPORT LIBRARIES
# ============================================================

import os
import pickle
import numpy as np
import pandas as pd

from xgboost import XGBClassifier

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

print("✅ All libraries imported successfully")
print("Current directory:", os.getcwd())

✅ All libraries imported successfully
Current directory: c:\Users\ashwi\OneDrive\Desktop\RiskPulse\notebooks


In [5]:
# ============================================================
# CELL 2: LOAD EXISTING PREPROCESSED DATA
# ============================================================

DATA_PATH = "../data/final_fraud_dataset.csv"

df = pd.read_csv(DATA_PATH)

print("✅ Dataset loaded successfully")
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

✅ Dataset loaded successfully
Shape: (529311, 433)

Columns:
['isFraud', 'TransactionDT', 'TransactionAmt', 'ProductCD', 'card1', 'card2', 'card3', 'card4', 'card5', 'card6', 'addr1', 'addr2', 'dist1', 'dist2', 'P_emaildomain', 'R_emaildomain', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14', 'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9', 'D10', 'D11', 'D12', 'D13', 'D14', 'D15', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'V29', 'V30', 'V31', 'V32', 'V33', 'V34', 'V35', 'V36', 'V37', 'V38', 'V39', 'V40', 'V41', 'V42', 'V43', 'V44', 'V45', 'V46', 'V47', 'V48', 'V49', 'V50', 'V51', 'V52', 'V53', 'V54', 'V55', 'V56', 'V57', 'V58', 'V59', 'V60', 'V61', 'V62', 'V63', 'V64', 'V65', 'V66', 'V67', 'V68', 'V69', 'V70', 'V71', 'V72', 'V73', 'V74', 'V75', 

,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,0,86400,68.5,4,13926,361.0,150.0,2,142.0,2,...,14,24.0,260,0,2,2,2,2,0,1557
1,0,86401,29.0,4,2755,404.0,150.0,3,102.0,2,...,14,24.0,260,0,2,2,2,2,0,1557
2,0,86469,59.0,4,4663,490.0,150.0,4,166.0,3,...,14,24.0,260,0,2,2,2,2,0,1557
3,0,86499,50.0,4,18132,567.0,150.0,3,117.0,3,...,14,24.0,260,0,2,2,2,2,0,1557
4,0,86506,50.0,1,4497,514.0,150.0,3,102.0,2,...,124,32.0,164,4,1,0,1,1,2,954


In [6]:
# ============================================================
# CELL 3: SEPARATE TARGET AND FEATURES
# ============================================================

TARGET = "isFraud"

# Target
y = df[TARGET].astype(int)

# Features
X = df.drop(columns=[TARGET])

print("✅ Target and features separated")
print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

print("\nFraud percentage:")
print(round(y.mean() * 100, 4), "%")

print("\nFeature count:", X.shape[1])

print("\nNon-numeric columns:")
print(X.select_dtypes(exclude=["number"]).columns.tolist())

✅ Target and features separated
X shape: (529311, 432)
y shape: (529311,)

Target distribution:
isFraud
0    510921
1     18390
Name: count, dtype: int64

Fraud percentage:
3.4743 %

Feature count: 432

Non-numeric columns:
[]


In [7]:
# ============================================================
# CELL 4: TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("✅ Train/Test split completed")

print("\nTraining data:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nTesting data:")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

print("\nTraining fraud rate:")
print(round(y_train.mean() * 100, 4), "%")

print("Testing fraud rate:")
print(round(y_test.mean() * 100, 4), "%")

✅ Train/Test split completed

Training data:
X_train: (423448, 432)
y_train: (423448,)

Testing data:
X_test: (105863, 432)
y_test: (105863,)

Training fraud rate:
3.4743 %
Testing fraud rate:
3.4743 %


In [8]:
# ============================================================
# CELL 5: CALCULATE CLASS WEIGHT
# ============================================================

negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()

scale_pos_weight = negative_count / positive_count

print("✅ Class weight calculated")

print("Normal transactions:", negative_count)
print("Fraud transactions :", positive_count)

print(
    "Scale positive weight:",
    round(scale_pos_weight, 4)
)

✅ Class weight calculated
Normal transactions: 408736
Fraud transactions : 14712
Scale positive weight: 27.7825


In [10]:
# ============================================================
# CELL 6: TRAIN XGBOOST MODEL
# ============================================================

from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

print("🚀 Starting XGBoost training...")
print("Training features:", X_train.shape[1])
print("Training rows:", X_train.shape[0])

model.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

print("\n✅ XGBoost training completed!")
print("Model feature count:", model.n_features_in_)
print("Trees trained:", model.get_booster().num_boosted_rounds())

🚀 Starting XGBoost training...
Training features: 432
Training rows: 423448

✅ XGBoost training completed!
Model feature count: 432
Trees trained: 500


In [11]:
# ============================================================
# CELL 7: TEST TRANSACTION AMOUNT SENSITIVITY
# ============================================================

# Take one real transaction from the test dataset
test_transaction = X_test.iloc[[0]].copy()

print("Original transaction amount:")
print(test_transaction["TransactionAmt"].iloc[0])

print("\nTesting different transaction amounts...")
print("-" * 50)

test_amounts = [
    500,
    2500,
    5000,
    12000,
    23000,
    50000,
    100000,
    500000
]

for amount in test_amounts:

    test_input = test_transaction.copy()

    # Change ONLY TransactionAmt
    test_input["TransactionAmt"] = amount

    probability = model.predict_proba(test_input)[0][1]

    print(
        f"₹{amount:>8,.0f}  →  "
        f"Probability: {probability:.6f}  |  "
        f"ML Score: {probability * 100:.2f}"
    )

Original transaction amount:
15.0

Testing different transaction amounts...
--------------------------------------------------
₹     500  →  Probability: 0.013842  |  ML Score: 1.38
₹   2,500  →  Probability: 0.011820  |  ML Score: 1.18
₹   5,000  →  Probability: 0.011820  |  ML Score: 1.18
₹  12,000  →  Probability: 0.011820  |  ML Score: 1.18
₹  23,000  →  Probability: 0.011820  |  ML Score: 1.18
₹  50,000  →  Probability: 0.011820  |  ML Score: 1.18
₹ 100,000  →  Probability: 0.011820  |  ML Score: 1.18
₹ 500,000  →  Probability: 0.011820  |  ML Score: 1.18


In [12]:
# ============================================================
# CELL 8: CHECK TRANSACTION AMOUNT DISTRIBUTION
# ============================================================

print("TransactionAmt statistics:")
print(X["TransactionAmt"].describe())

print("\nHighest transaction amounts:")
print(
    X["TransactionAmt"]
    .sort_values(ascending=False)
    .head(20)
    .to_list()
)

print("\nTransaction amounts above ₹1,000:")
print((X["TransactionAmt"] > 1000).sum())

print("\nTransaction amounts above ₹5,000:")
print((X["TransactionAmt"] > 5000).sum())

print("\nTransaction amounts above ₹10,000:")
print((X["TransactionAmt"] > 10000).sum())

TransactionAmt statistics:
count    529311.000000
mean        134.791403
std         237.960590
min           0.251000
25%          43.382000
50%          68.561000
75%         125.000000
max       31937.391000
Name: TransactionAmt, dtype: float64

Highest transaction amounts:
[31937.391, 31937.391, 6450.97, 6085.23, 5543.23, 5420.0, 5420.0, 5279.95, 5279.95, 5279.95, 5278.95, 5191.0, 5191.0, 5094.95, 5047.47, 4989.97, 4976.31, 4843.75, 4843.75, 4836.33]

Transaction amounts above ₹1,000:
6348

Transaction amounts above ₹5,000:
15

Transaction amounts above ₹10,000:
2


In [13]:
# ============================================================
# CELL 9: EVALUATE NEW XGBOOST MODEL
# ============================================================

y_pred_probability = model.predict_proba(X_test)[:, 1]
y_pred = (y_pred_probability >= 0.5).astype(int)

roc_auc = roc_auc_score(y_test, y_pred_probability)
pr_auc = average_precision_score(y_test, y_pred_probability)

print("=" * 60)
print("RISK PULSE MODEL EVALUATION")
print("=" * 60)

print(f"ROC-AUC : {roc_auc:.4f}")
print(f"PR-AUC  : {pr_auc:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

RISK PULSE MODEL EVALUATION
ROC-AUC : 0.9504
PR-AUC  : 0.7113

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.93      0.96    102185
           1       0.30      0.83      0.44      3678

    accuracy                           0.93    105863
   macro avg       0.65      0.88      0.70    105863
weighted avg       0.97      0.93      0.94    105863


Confusion Matrix:
[[95146  7039]
 [  627  3051]]


In [14]:
# ============================================================
# CELL 10: SAVE NEW RISK PULSE MODEL
# ============================================================

PROJECT_ROOT = os.path.dirname(os.getcwd())

MODEL_PATH = os.path.join(
    PROJECT_ROOT,
    "riskpulse_xgboost_model.pkl"
)

with open(MODEL_PATH, "wb") as f:
    pickle.dump(model, f)

print("=" * 60)
print("✅ NEW RISK PULSE MODEL SAVED")
print("=" * 60)

print("Model path:")
print(MODEL_PATH)

print("\nModel features:", model.n_features_in_)
print("Trees:", model.get_booster().num_boosted_rounds())

print("\nFile exists:", os.path.exists(MODEL_PATH))

✅ NEW RISK PULSE MODEL SAVED
Model path:
c:\Users\ashwi\OneDrive\Desktop\RiskPulse\riskpulse_xgboost_model.pkl

Model features: 432
Trees: 500

File exists: True


In [15]:
# ============================================================
# CELL 11: VERIFY SAVED MODEL
# ============================================================

with open(MODEL_PATH, "rb") as f:
    saved_model = pickle.load(f)

print("=" * 60)
print("SAVED MODEL VERIFICATION")
print("=" * 60)

print("Model type:", type(saved_model).__name__)
print("Features:", saved_model.n_features_in_)
print("Trees:", saved_model.get_booster().num_boosted_rounds())

print("\n✅ Saved model is the NEW 432-feature model")

SAVED MODEL VERIFICATION
Model type: XGBClassifier
Features: 432
Trees: 500

✅ Saved model is the NEW 432-feature model


In [16]:
# ============================================================
# CELL 12: SAVE MODEL METADATA
# ============================================================

model_metadata = {
    "feature_names": list(X.columns),
    "n_features": X.shape[1],
    "target": TARGET,
    "roc_auc": roc_auc,
    "pr_auc": pr_auc,
    "trees": model.get_booster().num_boosted_rounds()
}

print("=" * 60)
print("RISK PULSE MODEL METADATA")
print("=" * 60)

print("Features :", model_metadata["n_features"])
print("Target   :", model_metadata["target"])
print("ROC-AUC  :", round(model_metadata["roc_auc"], 4))
print("PR-AUC   :", round(model_metadata["pr_auc"], 4))
print("Trees    :", model_metadata["trees"])

print("\n✅ Metadata prepared")

RISK PULSE MODEL METADATA
Features : 432
Target   : isFraud
ROC-AUC  : 0.9504
PR-AUC   : 0.7113
Trees    : 500

✅ Metadata prepared


In [17]:
# ============================================================
# CELL 13: TEST REAL TRANSACTIONS
# ============================================================

# Get actual transactions from the test set
sample_indices = X_test.index[:10]

real_samples = X_test.loc[sample_indices]
real_labels = y_test.loc[sample_indices]

predicted_probabilities = model.predict_proba(real_samples)[:, 1]

results = pd.DataFrame({
    "Actual_isFraud": real_labels.values,
    "TransactionAmt": real_samples["TransactionAmt"].values,
    "Fraud_Probability": predicted_probabilities,
    "ML_Risk_Score": predicted_probabilities * 100
})

results["ML_Risk_Level"] = results["ML_Risk_Score"].apply(
    lambda x:
        "CRITICAL" if x >= 80 else
        "HIGH" if x >= 60 else
        "MEDIUM" if x >= 30 else
        "LOW"
)

display(results)

,Actual_isFraud,TransactionAmt,Fraud_Probability,ML_Risk_Score,ML_Risk_Level
0,0,15.00,0.005398,0.539798,LOW
1,0,50.00,0.152868,15.286830,LOW
2,0,57.95,0.079798,7.979784,LOW
3,0,117.00,0.201657,20.165726,LOW
4,0,49.00,0.205326,20.532578,LOW
5,0,250.00,0.096964,9.696414,LOW
6,0,34.00,0.171690,17.169025,LOW
7,0,46.04,0.188036,18.803621,LOW
8,0,25.00,0.063069,6.306870,LOW
9,0,159.95,0.038095,3.809521,LOW


In [18]:
# ============================================================
# CELL 14: TEST ACTUAL FRAUD TRANSACTIONS
# ============================================================

fraud_indices = X_test[y_test == 1].index[:10]

fraud_samples = X_test.loc[fraud_indices]
fraud_labels = y_test.loc[fraud_indices]

fraud_probabilities = model.predict_proba(
    fraud_samples
)[:, 1]

fraud_results = pd.DataFrame({
    "Actual_isFraud": fraud_labels.values,
    "TransactionAmt": fraud_samples["TransactionAmt"].values,
    "Fraud_Probability": fraud_probabilities,
    "ML_Risk_Score": fraud_probabilities * 100
})

fraud_results["ML_Risk_Level"] = fraud_results[
    "ML_Risk_Score"
].apply(
    lambda x:
        "CRITICAL" if x >= 80 else
        "HIGH" if x >= 60 else
        "MEDIUM" if x >= 30 else
        "LOW"
)

display(fraud_results)

,Actual_isFraud,TransactionAmt,Fraud_Probability,ML_Risk_Score,ML_Risk_Level
0,1,500.000,0.885968,88.596817,CRITICAL
1,1,38.944,0.997178,99.717804,CRITICAL
2,1,34.000,0.653877,65.387657,HIGH
3,1,944.000,0.825958,82.595757,CRITICAL
4,1,141.000,0.918812,91.881248,CRITICAL
5,1,26.585,0.986199,98.619911,CRITICAL
6,1,108.500,0.670254,67.025406,HIGH
7,1,117.000,0.854175,85.417480,CRITICAL
8,1,29.000,0.874127,87.412727,CRITICAL
9,1,87.000,0.797993,79.799324,HIGH


In [19]:
# ============================================================
# CELL 15: FINAL MODEL PREDICTION CHECK
# ============================================================

from sklearn.metrics import confusion_matrix, classification_report

# Predict fraud probabilities
test_probabilities = model.predict_proba(X_test)[:, 1]

# Convert probability to fraud/not-fraud
test_predictions = (test_probabilities >= 0.5).astype(int)

# Confusion matrix
cm = confusion_matrix(y_test, test_predictions)

print("=" * 60)
print("RISK PULSE FINAL PREDICTION CHECK")
print("=" * 60)

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        test_predictions,
        digits=4
    )
)

RISK PULSE FINAL PREDICTION CHECK

Confusion Matrix:
[[95146  7039]
 [  627  3051]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9935    0.9311    0.9613    102185
           1     0.3024    0.8295    0.4432      3678

    accuracy                         0.9276    105863
   macro avg     0.6479    0.8803    0.7022    105863
weighted avg     0.9694    0.9276    0.9433    105863



In [20]:
# ============================================================
# CELL 16: FINAL MODEL SAVE
# ============================================================

import os
import pickle

PROJECT_ROOT = os.path.dirname(os.getcwd())

MODEL_PATH = os.path.join(
    PROJECT_ROOT,
    "riskpulse_xgboost_model.pkl"
)

with open(MODEL_PATH, "wb") as f:
    pickle.dump(model, f)

print("=" * 60)
print("✅ FINAL RISK PULSE MODEL SAVED")
print("=" * 60)

print("Path:", MODEL_PATH)
print("Features:", model.n_features_in_)
print("Trees:", model.get_booster().num_boosted_rounds())
print("Exists:", os.path.exists(MODEL_PATH))

✅ FINAL RISK PULSE MODEL SAVED
Path: c:\Users\ashwi\OneDrive\Desktop\RiskPulse\riskpulse_xgboost_model.pkl
Features: 432
Trees: 500
Exists: True


In [2]:
print("X exists:", "X" in globals())
print("y exists:", "y" in globals())
print("X_train exists:", "X_train" in globals())
print("X_test exists:", "X_test" in globals())
print("model exists:", "model" in globals())

X exists: False
y exists: False
X_train exists: False
X_test exists: False
model exists: False


In [3]:
import os

print(os.getcwd())
print(os.listdir(os.path.dirname(os.getcwd())))

c:\Users\ashwi\OneDrive\Desktop\RiskPulse\notebooks
['.venv', 'backend', 'dashboard', 'data', 'frontend', 'notebooks', 'README.md', 'riskpulse_xgboost_model.pkl']


In [6]:
import os
import pandas as pd
import numpy as np
import pickle

print("Current working directory:")
print(os.getcwd())

print("\nProject root:")
PROJECT_ROOT = os.path.dirname(os.getcwd())
print(PROJECT_ROOT)

print("\nData folder:")
DATA_FOLDER = os.path.join(PROJECT_ROOT, "data")
print(DATA_FOLDER)

print("\nFiles in data folder:")
print(os.listdir(DATA_FOLDER))

Current working directory:
c:\Users\ashwi\OneDrive\Desktop\RiskPulse\notebooks

Project root:
c:\Users\ashwi\OneDrive\Desktop\RiskPulse

Data folder:
c:\Users\ashwi\OneDrive\Desktop\RiskPulse\data

Files in data folder:
['final_fraud_dataset.csv', 'IEEE-CIS-Fraud-Detection.zip']


In [8]:
# ============================================================
# CELL 2 — LOAD EXISTING PREPROCESSED DATASET
# ============================================================

import pandas as pd
import numpy as np
import os

PROJECT_ROOT = os.path.dirname(os.getcwd())

DATA_FILE = os.path.join(
    PROJECT_ROOT,
    "data",
    "final_fraud_dataset.csv"
)

df = pd.read_csv(DATA_FILE)

print("=" * 60)
print("✅ EXISTING PREPROCESSED DATASET LOADED")
print("=" * 60)

print("Dataset shape:", df.shape)
print("Target column:", "isFraud")
print("Fraud transactions:", int(df["isFraud"].sum()))
print("Normal transactions:", int((df["isFraud"] == 0).sum()))

✅ EXISTING PREPROCESSED DATASET LOADED
Dataset shape: (529311, 433)
Target column: isFraud
Fraud transactions: 18390
Normal transactions: 510921


In [9]:
# ============================================================
# CELL 3 — TARGET AND FEATURES
# ============================================================

TARGET = "isFraud"

X = df.drop(columns=[TARGET])
y = df[TARGET]

print("=" * 60)
print("✅ TARGET AND FEATURES CREATED")
print("=" * 60)

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

print("\nFraud percentage:",
      round(y.mean() * 100, 4), "%")

print("\nFeature count:", X.shape[1])

non_numeric = X.select_dtypes(
    exclude=[np.number]
).columns.tolist()

print("\nNon-numeric columns:", non_numeric)

✅ TARGET AND FEATURES CREATED
X shape: (529311, 432)
y shape: (529311,)

Target distribution:
isFraud
0    510921
1     18390
Name: count, dtype: int64

Fraud percentage: 3.4743 %

Feature count: 432

Non-numeric columns: []


In [10]:
# Find a normal transaction close to ₹499

normal_rows = df[df["isFraud"] == 0].copy()

normal_rows["amount_difference"] = (
    normal_rows["TransactionAmt"] - 499
).abs()

normal_row = normal_rows.sort_values(
    "amount_difference"
).iloc[0]

print("✅ Normal reference transaction found")
print("Original amount:", normal_row["TransactionAmt"])
print("Actual fraud:", normal_row["isFraud"])

✅ Normal reference transaction found
Original amount: 498.95
Actual fraud: 0.0


In [11]:
# Create complete 432-feature transaction
test_transaction = normal_row.drop(
    labels=["isFraud", "amount_difference"]
).copy()

# Change ONLY the transaction amount
test_transaction["TransactionAmt"] = 499.0

print("✅ Test transaction created")
print("TransactionAmt:", test_transaction["TransactionAmt"])
print("Number of features:", len(test_transaction))

✅ Test transaction created
TransactionAmt: 499.0
Number of features: 432


In [13]:
# ============================================================
# LOAD THE ALREADY TRAINED MODEL
# ============================================================

import os
import pickle

PROJECT_ROOT = os.path.dirname(os.getcwd())

MODEL_FILE = os.path.join(
    PROJECT_ROOT,
    "riskpulse_xgboost_model.pkl"
)

with open(MODEL_FILE, "rb") as f:
    model = pickle.load(f)

print("✅ Existing model loaded")
print("Model features:", model.n_features_in_)

✅ Existing model loaded
Model features: 432


In [14]:
# Find a normal transaction close to ₹499

normal_rows = df[df["isFraud"] == 0].copy()

normal_rows["amount_difference"] = (
    normal_rows["TransactionAmt"] - 499
).abs()

normal_row = normal_rows.sort_values(
    "amount_difference"
).iloc[0]

print("✅ Normal reference transaction found")
print("Original amount:", normal_row["TransactionAmt"])
print("Actual fraud:", normal_row["isFraud"])


# Create complete transaction
test_transaction = normal_row.drop(
    labels=["isFraud", "amount_difference"]
).copy()

# Change ONLY amount
test_transaction["TransactionAmt"] = 499.0

test_df = pd.DataFrame([test_transaction])

probability = model.predict_proba(test_df)[0][1]

print("=" * 50)
print("₹499 NORMAL-PROFILE TEST")
print("=" * 50)
print("Features:", test_df.shape[1])
print("Fraud probability:", round(probability, 6))
print("ML risk score:", round(probability * 100, 2))

✅ Normal reference transaction found
Original amount: 498.95
Actual fraud: 0.0
₹499 NORMAL-PROFILE TEST
Features: 432
Fraud probability: 0.204328
ML risk score: 20.43
